In [1]:
import os
import sys
os.chdir('/zhome/71/c/146676/texture_tomography')
sys.path.append('/zhome/71/c/146676/texture_tomography/package/odf_mumott')
import numpy as np

from package.cil_addons.texture_tomography.operators.pfo_and_projection_batched_opencl import PFO_OPENCL_BATCHED
from package.cil_addons.texture_tomography.operators.operator_memory_model import OperatorMemoryModel
from package.cil_addons.texture_tomography.operators.memory_tracker import MemoryCounter
from package.cil_addons.texture_tomography.optimization.fista_opencl import FISTAOpenCL
from package.cil_addons.texture_tomography.optimization.fista_memory_model import FISTAMemoryModel

import matplotlib.pyplot as plt

import yaml

INFO:Setting the number of threads to 8. If your physical cores are fewer than this number, you may want to use numba.set_num_threads(n), and os.environ["OPENBLAS_NUM_THREADS"] = f"{n}" to set the number of threads to the number of physical cores n.
INFO:Setting numba log level to WARNING.


In [2]:
config_path = "configs/aluminum_config_small.yaml"
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

N_theta = 100
two_thetas = np.linspace(0.01,0.6,N_theta)

op = PFO_OPENCL_BATCHED(
        cfg = cfg,
        two_thetas = two_thetas,
        verbose=False)
op.set_pf_batch_max_gb(3.0)

# mem = MemoryCounter()
# memory_model = OperatorMemoryModel(op, mem)
# memory_model.mem.report(show_peak=True)
# memory_model.mem.report_allocations()
# memory_model.model_direct()
# memory_model.mem.report(show_peak=True)
# memory_model.mem.report_allocations()
# memory_model.model_adjoint()
# memory_model.mem.report(show_peak=True)
# memory_model.mem.report_allocations()

In [3]:
mem = MemoryCounter()

op_model = OperatorMemoryModel(op, mem)

fista = FISTAOpenCL(
    operator=op,
    prox_kind="nonneg_tv",
    lam=0.0,
    tau=1e-3
)

fista_mem = FISTAMemoryModel(fista, mem, op_model)

x_shape  = (op.Nx, op.Nx, op.K_sum)
Ax_shape = (op.N_rot, op.Nx, op.N_chi * op.N_theta)

fista_mem.model_run(x_shape, Ax_shape, niter=1)

mem.report()
mem.report_peak_allocations(min_mb=50)


=== MemoryCounter Report ===
Current : 5.880 GB
Peak    : 14.834 GB
Live allocations : 21

=== Peak Allocation Breakdown ===
Name                                        Size (MB)
-------------------------------------------------------
B_gpu_batch[m0,b0]                           3071.365
BT_gpu_batch[m0,b0]                          3071.365
fista.Ax                                     1524.353
fista.r                                      1524.353
pf_basis_batch[m0,b0]                        1228.546
_out_sub[0]                                   457.306
data_gpu_sub[m0]                              457.306
_coeffs_t_gpu[0]                              390.742
_coeffs_gpu_full_sino                         390.742
_x_full_gpu                                   390.742
xin_gpu[m0]                                   390.742
fista.x                                       192.766
fista.y                                       192.766
fista.x_old                                   192.766
fista.v  